# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/gopikadevabalaji/world-population/world_population.csv


In [2]:

import pandas as pd
import glob

path = glob.glob('/kaggle/input/datasets/gopikadevabalaji/world-population/world_population.csv', recursive=True)[0]

df = pd.read_csv(path)

display(df.head())


,Rank,CCA3,Country/Territory,Capital,Continent,2022 Population,2020 Population,2015 Population,2010 Population,2000 Population,1990 Population,1980 Population,1970 Population,Area (km²),Density (per km²),World Population Percentage
0,36,AFG,Afghanistan,Kabul,Asia,41128771,38972230,33753499,28189672,19542982,10694796,12486631,10752971,652230,63.0587,0.52
1,138,ALB,Albania,Tirana,Europe,2842321,2866849,2882481,2913399,3182021,3295066,2941651,2324731,28748,98.8702,0.04
2,34,DZA,Algeria,Algiers,Africa,44903225,43451666,39543154,35856344,30774621,25518074,18739378,13795915,2381741,18.8531,0.56
3,213,ASM,American Samoa,Pago Pago,Oceania,44273,46189,51368,54849,58230,47818,32886,27075,199,222.4774,0.00
4,203,AND,Andorra,Andorra la Vella,Europe,79824,77700,71746,71519,66097,53569,35611,19860,468,170.5641,0.00


In [3]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('/kaggle/input/datasets/gopikadevabalaji/world-population/world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


In [4]:

# Colour-blind-friendly palette (Okabe-Ito)

COLORS = {
    'Africa': '#E69F00',         # Orange
    'Asia': '#0072B2',           # Blue
    'Europe': '#CC79A7',         # Purple
    'North America': '#009E73',  # Bluish green
    'South America': '#D55E00',  # Vermilion
    'Oceania': '#56B4E9',        # Sky blue
    'Rest of the world': '#999999'  # Grey
}


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart Type: Line Chart
- A line chart is suitable for showing population growth trends from 1970 to 2022.
- Using 1970 as a baseline (index = 100) allows fair comparison between continents of different population sizes.
- A colour-blind-friendly categorical palette helps distinguish continents.
- Markers and annotations highlight important changes and the fastest-growing continent.
- A clean white background and minimal gridlines improve readability.

In [5]:
# Q1

import pandas as pd
import plotly.express as px


COLORS = {
    'Africa': '#E69F00',
    'Asia': '#0072B2',
    'Europe': '#CC79A7',
    'North America': '#009E73',
    'South America': '#D55E00',
    'Oceania': '#56B4E9',
    'Rest of the world': '#999999'
}


continent_year = (
    df_long.groupby(['Continent', 'Year'], as_index=False)
    ['Population'].sum()
)


baseline = (
    continent_year[continent_year['Year'] == 1970]
    [['Continent', 'Population']]
    .rename(columns={'Population': 'Base'})
)


growth = continent_year.merge(baseline, on='Continent')

growth['Index'] = (
    growth['Population'] / growth['Base'] * 100
)


latest = growth[growth['Year'] == 2022]
winner = latest.loc[latest['Index'].idxmax()]


fig = px.line(
    growth,
    x='Year',
    y='Index',
    color='Continent',
    color_discrete_map=COLORS,
    markers=True,
    title=f"{winner['Continent']} experienced the fastest population growth since 1970",
    labels={'Index': 'Population index (1970 = 100)'}
)


fig.update_traces(
    line=dict(width=3),
    marker=dict(size=7)
)


fig.add_hline(
    y=100,
    line_dash='dot',
    line_color='gray',
    line_width=2
)


fig.add_annotation(
    x=1980,
    y=100,
    text='1970 baseline (100)',
    showarrow=False,
    xanchor='left',
    yanchor='top',
    yshift=-8,
    font=dict(size=11, color='gray')
)


fig.add_annotation(
    x=2022,
    y=winner['Index'],
    text=f"{winner['Index']/100:.1f}× 1970 population",
    showarrow=True,
    arrowhead=2,
    ax=-110,
    ay=-45,
    font=dict(size=12, color='#333333'),
    bgcolor='white',
    bordercolor='#DDDDDD',
    borderpad=5
)


fig.update_layout(
    template='plotly_white',
    height=600,
    title_x=0.03,
    legend_title_text='',
    hovermode='x unified',
    margin=dict(l=70, r=80, t=100, b=80)
)



fig.update_xaxes(
    title=None,
    tickvals=[1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022],
    tickangle=-45,
    showgrid=False,
    tickfont=dict(size=11)
)



fig.update_yaxes(
    title='Population index (1970 = 100)',
    range=[80, 450],
    showgrid=True,
    gridcolor='#EEEEEE'
)

fig.show()



---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart Type: Horizontal Bar Chart
- A horizontal bar chart clearly compares the population increases of the top 15 countries.
- Sorting the bars makes it easy to identify countries with the largest increases.
- Orange highlights African countries, while grey provides context for other countries.
- Direct value labels reduce the need to estimate values from the axis.
- Minimal gridlines and a clean layout reduce visual clutter.

In [6]:
# Q2

df['Population Increase'] = (
    df['2022 Population'] - df['1970 Population']
)

top15 = df.nlargest(15, 'Population Increase').copy()
top15 = top15.sort_values('Population Increase')

africa_count = (top15['Continent'] == 'Africa').sum()

top15['Highlight'] = np.where(
    top15['Continent'] == 'Africa',
    'Africa',
    'Other continents'
)

fig = px.bar(
    top15,
    x='Population Increase',
    y='Country',
    orientation='h',
    color='Highlight',
    color_discrete_map={
        'Africa': '#E69F00',
        'Other continents': '#BDBDBD'
    },
    text='Population Increase',
    title=f"{africa_count} African countries rank among the 15 largest population increases"
)

fig.update_traces(
    texttemplate='%{text:.3s}',
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    template='plotly_white',
    height=650,
    title_x=0.03,
    showlegend=False,
    bargap=0.25,
    margin=dict(r=100)
)

fig.update_xaxes(
    title='Population increase (1970–2022)',
    showgrid=True,
    gridcolor='#EEEEEE'
)

fig.update_yaxes(title=None, showgrid=False)

fig.show()


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart Type: Scatter Plot
- A scatter plot is appropriate for examining the relationship between two numerical variables: land area and population.
- Logarithmic scales help display countries with very different population sizes and land areas.
- Different colours represent continents, making regional patterns easier to identify.
- Annotations highlight important outliers such as India, China, Russia and Australia.
- Tooltips provide additional information without overcrowding the chart.

In [7]:
# Q3

scatter_df = df[
    (df['Area'] > 0) &
    (df['2022 Population'] > 0)
].copy()

fig = px.scatter(
    scatter_df,
    x='Area',
    y='2022 Population',
    color='Continent',
    color_discrete_map=COLORS,
    hover_name='Country',
    hover_data={
        'Area': ':,',
        '2022 Population': ':,',
        'Continent': True
    },
    log_x=True,
    log_y=True,
    opacity=0.75,
    title='Land area alone does not determine population size'
)

fig.update_traces(
    marker=dict(size=9, line=dict(width=0.5, color='white'))
)

# Annotate important examples
for country in ['India', 'China', 'Russia', 'Bangladesh', 'Australia']:
    row = scatter_df[scatter_df['Country'] == country]
    if not row.empty:
        fig.add_annotation(
            x=row.iloc[0]['Area'],
            y=row.iloc[0]['2022 Population'],
            text=country,
            showarrow=True,
            arrowhead=1,
            ax=35,
            ay=-25,
            font=dict(size=11)
        )

fig.update_layout(
    template='plotly_white',
    height=600,
    title_x=0.03,
    legend_title_text='',
    legend=dict(orientation='h', y=-0.2)
)

fig.update_xaxes(
    title='Land area (km², log scale)',
    showgrid=True,
    gridcolor='#EEEEEE'
)

fig.update_yaxes(
    title='2022 population (log scale)',
    showgrid=True,
    gridcolor='#EEEEEE'
)

fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart Type: Box Plot
- A box plot is useful for comparing the distribution of population density across continents.
- It displays the median, interquartile range and outliers rather than only average values.
- Sorting continents by median density makes comparisons easier.
- A logarithmic scale helps display both low-density and high-density countries.
- Consistent continent colours and a minimal design improve readability.

In [8]:
# Q4

continent_order = (
    df.groupby('Continent')['Density']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    color_discrete_map=COLORS,
    category_orders={'Continent': continent_order},
    points='all',
    hover_name='Country',
    log_y=True,
    title='Population density varies widely within and across continents'
)

fig.update_traces(
    jitter=0.35,
    pointpos=0,
    marker=dict(size=4, opacity=0.35),
    line=dict(width=1.5)
)

fig.update_layout(
    template='plotly_white',
    height=580,
    title_x=0.03,
    showlegend=False
)

fig.update_xaxes(
    title=None,
    showgrid=False
)

fig.update_yaxes(
    title='Population density (people/km², log scale)',
    showgrid=True,
    gridcolor='#EEEEEE'
)

fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart Type: Choropleth Map
- A choropleth map is suitable for visualizing geographical patterns of population growth and decline.
- A diverging colour palette distinguishes countries experiencing population decline from those experiencing growth.
- A neutral midpoint at zero makes the direction of population change easier to interpret.
- However, information provides exact growth percentages for individual countries.
- Country borders and a simple map design help identify regional patterns without unnecessary visual clutter.

In [9]:
# Q5

df['Growth 2000-2022 (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population'] * 100
)

# Symmetric scale around zero
limit = np.percentile(
    abs(df['Growth 2000-2022 (%)']), 95
)

fig = px.choropleth(
    df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Growth 2000-2022 (%)',
    hover_name='Country',
    hover_data={'Growth 2000-2022 (%)': ':.1f'},
    color_continuous_scale=[
        [0.0, '#D55E00'],
        [0.5, '#F2F2F2'],
        [1.0, '#0072B2']
    ],
    range_color=[-limit, limit],
    color_continuous_midpoint=0,
    title='Population growth and decline show distinct geographic patterns'
)

fig.update_geos(
    showcoastlines=False,
    showframe=False,
    showcountries=True,
    countrycolor='white',
    projection_type='natural earth',
    showland=True,
    landcolor='#EAEAEA'
)

fig.update_layout(
    template='plotly_white',
    height=600,
    title_x=0.03,
    coloraxis_colorbar=dict(
        title='Change (%)',
        ticksuffix='%'
    ),
    margin=dict(l=10, r=10, t=80, b=20)
)

fig.show()

# Check countries with declining populations
display(
    df[df['Growth 2000-2022 (%)'] < 0]
    [['Country', 'Growth 2000-2022 (%)']]
    .sort_values('Growth 2000-2022 (%)')
    .head(10)
)


,Country,Growth 2000-2022 (%)
153,Northern Mariana Islands,-38.321840
3,American Samoa,-23.968745
117,Lithuania,-23.601880
126,Marshall Islands,-23.338374
133,Moldova,-23.016822
111,Latvia,-22.648786
25,Bosnia and Herzegovina,-22.630888
226,Vatican City,-21.658986
229,Wallis and Futuna,-21.401888
218,Ukraine,-18.776723


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart Type: Treemap
- A treemap effectively shows the hierarchical distribution of population across continents and countries.
- Rectangle sizes represent population, allowing quick identification of the largest contributors.
- Distinct continent colours help separate geographical groups.
- Percentage labels and other information provide additional details about population shares.
- The space-efficient layout displays many countries without requiring multiple charts.

In [10]:
# Q6

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_map=COLORS,
    title='Asia accounts for the largest share of the world population'
)

fig.update_traces(
    textinfo='label+percent root',
    textfont=dict(size=13),
    marker=dict(line=dict(color='white', width=1)),
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Population: %{value:,.0f}<br>'
        'World share: %{percentRoot:.1%}'
        '<extra></extra>'
    )
)

fig.update_layout(
    template='plotly_white',
    height=650,
    title_x=0.03,
    margin=dict(l=20, r=20, t=80, b=20)
)

fig.show()


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart Type: 100% Stacked Bar Chart
- A 100% stacked bar chart is suitable for comparing changes in population proportions over time.
- Each bar represents 100% of the world population, making relative comparisons easier.
- Consistent colours distinguish Africa, Asia, Europe and the rest of the world.
- Percentage labels help readers understand each group's contribution.
- The chart highlights how the global population distribution has changed between 1970 and 2022.

In [11]:
# Q7

shares = (
    df_long.groupby(['Year', 'Continent'], as_index=False)
    ['Population'].sum()
)

shares['Group'] = shares['Continent'].where(
    shares['Continent'].isin(['Africa', 'Asia', 'Europe']),
    'Rest of the world'
)

shares = (
    shares.groupby(['Year', 'Group'], as_index=False)
    ['Population'].sum()
)

shares['World Share (%)'] = (
    shares['Population']
    / shares.groupby('Year')['Population'].transform('sum')
    * 100
)

group_order = [
    'Asia',
    'Africa',
    'Europe',
    'Rest of the world'
]

fig = px.bar(
    shares,
    x='Year',
    y='World Share (%)',
    color='Group',
    barmode='stack',
    category_orders={'Group': group_order},
    color_discrete_map=COLORS,
    text='World Share (%)',
    title='Africa gains world population share while Europe loses ground'
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(size=10)
)

fig.update_layout(
    template='plotly_white',
    height=600,
    title_x=0.03,
    legend_title_text='',
    legend=dict(orientation='h', y=-0.2),
    bargap=0.2
)

fig.update_xaxes(
    type='category',
    title=None,
    showgrid=False
)

fig.update_yaxes(
    title='Share of world population (%)',
    range=[0, 100],
    ticksuffix='%',
    showgrid=False
)

fig.show()

# Exact shares in 1970 and 2022
display(
    shares[shares['Year'].isin([1970, 2022])]
    .pivot(index='Group', columns='Year', values='World Share (%)')
    .round(2)
)


Year,1970,2022
Group,,
Africa,9.89,17.89
Asia,58.06,59.21
Europe,17.76,9.32
Rest of the world,14.29,13.57
